# 04. Feature Engineering - Automobile Loan Default Prediction

Group rare categories, derive new features, and select final features, building on `03_data_preprocessing.ipynb`'s output.


## 1. Setup


In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

processed_dir = 'data/processed' if os.path.exists('data/processed') else '../data/processed'

train_df = pd.read_csv(os.path.join(processed_dir, 'train_processed.csv'))
test_df = pd.read_csv(os.path.join(processed_dir, 'test_processed.csv'))

print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

train_df shape: (97484, 61)
test_df shape: (24372, 61)


## 2. Group Rare Categories

Categories below 1% of training rows get grouped into `"Other"`. Threshold fit on train only, applied to both.


In [2]:
cols_to_group = ['Type_Organization', 'Client_Education', 'Client_Income_Type', 'Client_Occupation']
min_count = len(train_df) * 0.01

for col in cols_to_group:
    value_counts = train_df[col].value_counts()
    rare_categories = value_counts[value_counts < min_count].index.tolist()
    train_df[col] = train_df[col].replace(rare_categories, 'Other')
    test_df[col] = test_df[col].replace(rare_categories, 'Other')
    print(f"{col}: grouped {len(rare_categories)} rare categories -> {train_df[col].nunique()} categories remain")

Type_Organization: grouped 41 rare categories -> 18 categories remain
Client_Education: grouped 1 rare categories -> 6 categories remain


Client_Income_Type: grouped 4 rare categories -> 6 categories remain
Client_Occupation: grouped 7 rare categories -> 13 categories remain


## 3. Encode Grouped Columns

One-hot encode the 4 now-grouped columns, fit on train, align test.


In [3]:
train_df = pd.get_dummies(train_df, columns=cols_to_group, drop_first=True)
test_df = pd.get_dummies(test_df, columns=cols_to_group, drop_first=True)

train_df, test_df = train_df.align(test_df, join='left', axis=1, fill_value=0)

dummy_cols = [c for c in train_df.columns if train_df[c].dtype == bool]
train_df[dummy_cols] = train_df[dummy_cols].astype(int)
test_df[dummy_cols] = test_df[dummy_cols].astype(int)

print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

train_df shape: (97484, 96)
test_df shape: (24372, 96)
